SETUP WORKSPACE:

In [1]:
# SET HYPERPARAMETERS

In [2]:
# IMPORT LIBRARIES

import pandas as pd

EXPLORATORY DATA ANALYSIS:

In [3]:
# LOAD DATA

health = pd.read_csv("user_health_data.csv")
supplements = pd.read_csv("supplement_usage.csv")
experiments = pd.read_csv("experiments.csv")
profiles = pd.read_csv("user_profiles.csv")

In [4]:
# SUMMARY OF EACH TABLE

# Get list of all tables
files = [health,
supplements,
experiments,
profiles]
# Iterate over the list
for file in files:
    df = file
    print("\n" + "=" * 70)
    print(file)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("\nDtypes:")
    print(df.dtypes)
    print("\nSample:")
    print(df.head())
    print("\nMissing:")
    print(df.isna().sum()) 


                                   user_id        date  average_heart_rate  \
0     c6ae338a-9f95-481c-a88d-24a58bc8fc71  2018-01-31           93.055612   
1     c6ae338a-9f95-481c-a88d-24a58bc8fc71  2018-02-28           88.059964   
2     c6ae338a-9f95-481c-a88d-24a58bc8fc71  2018-03-31           78.373746   
3     c6ae338a-9f95-481c-a88d-24a58bc8fc71  2018-04-30           62.204061   
4     5346f1dc-30f7-4e3a-9d35-eec6cb8835fa  2018-01-31           77.075789   
...                                    ...         ...                 ...   
1995  9c083dc3-b789-4cee-87fb-9887ff4b4e29  2018-04-30           88.575502   
1996  2f2aa322-d2b2-4c88-aeb6-e49e768521f3  2018-01-31           96.665724   
1997  2f2aa322-d2b2-4c88-aeb6-e49e768521f3  2018-02-28           88.219866   
1998  2f2aa322-d2b2-4c88-aeb6-e49e768521f3  2018-03-31           86.871171   
1999  2f2aa322-d2b2-4c88-aeb6-e49e768521f3  2018-04-30           75.773646   

      average_glucose sleep_hours  activity_level  
0         

In [5]:
# SEARCH FOR DUPLICATES

# Health table
print("Health duplicate user/date:",
    health.duplicated(["user_id", "date"]).sum())
# Supplement table
print("Supplement duplicate user/date:",
    supplements.duplicated(["user_id", "date"]).sum())

Health duplicate user/date: 0
Supplement duplicate user/date: 721


In [6]:
# INSPECT DATES

# Health table
print("Health dates:")
print(health["date"].value_counts().sort_index())
health_dates = set(health["date"])
print("Health dates:", sorted(health_dates))
# Supplement table
print("\nSupplement dates:")
print(supplements["date"].value_counts().sort_index())
supplement_dates = set(supplements["date"])
print("Supplement dates:", sorted(supplement_dates))
# Both tables relations
print("\nCommon:", sorted(health_dates & supplement_dates))
print("\nSupplement only:", sorted(supplement_dates - health_dates))
print("\nHealth only:", sorted(health_dates - supplement_dates))

Health dates:
date
2018-01-31    500
2018-02-28    500
2018-03-31    500
2018-04-30    500
Name: count, dtype: int64
Health dates: ['2018-01-31', '2018-02-28', '2018-03-31', '2018-04-30']

Supplement dates:
date
2018-01-31    511
2018-02-28    491
2018-03-31    508
2018-04-30    490
Name: count, dtype: int64
Supplement dates: ['2018-01-31', '2018-02-28', '2018-03-31', '2018-04-30']

Common: ['2018-01-31', '2018-02-28', '2018-03-31', '2018-04-30']

Supplement only: []

Health only: []


In [7]:
# CHECK IMPORTANT CATEGORIES

# Find unique supplements
print("Supplement names:")
print(supplements["supplement_name"].value_counts())
# Find unique dosage units
print("\nDosage units:")
print(supplements["dosage_unit"].value_counts())
# Find unique experiments
print("\nExperiment names:")
print(experiments["name"].value_counts())
# Inspect age of patients
print("\nAge range:")
print(profiles["age"].describe())

Supplement names:
supplement_name
Zinc         437
Vitamin C    404
Magnesium    395
Placebo      386
Omega-3      378
Name: count, dtype: int64

Dosage units:
dosage_unit
mg    2000
Name: count, dtype: int64

Experiment names:
name
Endurance        1
Strength         1
Sleep Quality    1
Focus            1
Memory           1
Recovery         1
Name: count, dtype: int64

Age range:
count    450.000000
mean      44.084444
std       15.411562
min       18.000000
25%       30.000000
50%       45.000000
75%       57.000000
max       69.000000
Name: age, dtype: float64


In [8]:
# TWO FINAL TESTS

# Check granularity and density/sparsity
print(supplements.groupby(["user_id", "date"])
    .size()
    .value_counts()
    .sort_index())
# Inspect closer healt table and supplements table relations via users and dates
merged_test = health.merge(
    supplements,
    on=["user_id", "date"],
    how="left"
)
# Show results
print("Health rows:", len(health))
print("Supplement rows:", len(supplements))
print("After left join:", len(merged_test))
print("Health rows with no supplement:",
      merged_test["supplement_name"].isna().sum())

1    766
2    353
3    119
4     35
5      5
6      1
Name: count, dtype: int64
Health rows: 2000
Supplement rows: 2000
After left join: 2721
Health rows with no supplement: 721


THE MERGE:

In [9]:
# MERGE THEM ALL

# DEFINE FUNCTION TO JOIN THE TABLES: We are not going to use anything from the EDA part we create the function that joins the tables from scratch
# Hence this is the solution, however keeping the EDA may be interesting for potential reviewer
def merge_all_data(
    health_file,
    supplement_file,
    experiments_file,
    profiles_file
):
    # Load data
    health = pd.read_csv(health_file)
    supplements = pd.read_csv(supplement_file)
    experiments = pd.read_csv(experiments_file)
    profiles = pd.read_csv(profiles_file)
    # Clean health data
    health["date"] = pd.to_datetime(health["date"])
    # Convert values such as "8.8h" and "8.0H" to numeric
    health["sleep_hours"] = pd.to_numeric(
        health["sleep_hours"]
        .astype(str)
        .str.replace(r"[hH]$", "", regex=True),
        errors="coerce"
    )
    # Clean supplement data
    supplements["date"] = pd.to_datetime(supplements["date"])

    # Convert dosage units from mg to g
    supplements["dosage_grams"] = supplements["dosage"] / 1000
    # Create age groups
    age_bins = [-float("inf"), 17, 25, 35, 45, 55, 65, float("inf")]
    age_labels = [
        "Under 18",
        "18-25",
        "26-35",
        "36-45",
        "46-55",
        "56-65",
        "Over 65"
    ]
    profiles["user_age_group"] = pd.cut(
        profiles["age"],
        bins=age_bins,
        labels=age_labels
    ).astype("object")
    profiles["user_age_group"] = profiles["user_age_group"].fillna("Unknown")
    # Add experiment names
    supplements = supplements.merge(
        experiments[["experiment_id", "name"]],
        on="experiment_id",
        how="left"
    )
    supplements = supplements.rename(
        columns={"name": "experiment_name"}
    )
    # Main merge
    result = health.merge(
        supplements[
            [
                "user_id",
                "date",
                "supplement_name",
                "dosage_grams",
                "is_placebo",
                "experiment_name"
            ]
        ],
        on=["user_id", "date"],
        how="left"
    )
    # Add profile information
    result = result.merge(
        profiles[
            [
                "user_id",
                "email",
                "user_age_group"
            ]
        ],
        on="user_id",
        how="left"
    )
    # Days without supplement intake
    result["supplement_name"] = result["supplement_name"].fillna("No intake")
    # Required output columns
    result = result[
        [
            "user_id",
            "date",
            "email",
            "user_age_group",
            "experiment_name",
            "supplement_name",
            "dosage_grams",
            "is_placebo",
            "average_heart_rate",
            "average_glucose",
            "sleep_hours",
            "activity_level"
        ]
    ]

    return result

In [ ]:
# TEST

# Apply the function
result = merge_all_data(
    'user_health_data.csv',
    'supplement_usage.csv',
    'experiments.csv',
    'user_profiles.csv'
)
# Save the dataframe
result.to_csv("solution.csv")
# Show results
print(result)

                                   user_id       date                   email  \
0     c6ae338a-9f95-481c-a88d-24a58bc8fc71 2018-01-31        hi_1@example.com   
1     c6ae338a-9f95-481c-a88d-24a58bc8fc71 2018-02-28        hi_1@example.com   
2     c6ae338a-9f95-481c-a88d-24a58bc8fc71 2018-03-31        hi_1@example.com   
3     c6ae338a-9f95-481c-a88d-24a58bc8fc71 2018-03-31        hi_1@example.com   
4     c6ae338a-9f95-481c-a88d-24a58bc8fc71 2018-04-30        hi_1@example.com   
...                                    ...        ...                     ...   
2716  9c083dc3-b789-4cee-87fb-9887ff4b4e29 2018-04-30  contact499@myemail.com   
2717  2f2aa322-d2b2-4c88-aeb6-e49e768521f3 2018-01-31   hello_500@myemail.com   
2718  2f2aa322-d2b2-4c88-aeb6-e49e768521f3 2018-02-28   hello_500@myemail.com   
2719  2f2aa322-d2b2-4c88-aeb6-e49e768521f3 2018-03-31   hello_500@myemail.com   
2720  2f2aa322-d2b2-4c88-aeb6-e49e768521f3 2018-04-30   hello_500@myemail.com   

     user_age_group experim

In [11]:
# ADDITIONAL INSPECTION OF THE RESULTS

print(result.shape)
print(result.columns.tolist())
print(result.head())
print(result.isna().sum())
print(result["user_age_group"].value_counts(dropna=False))
print(result["supplement_name"].value_counts())
print(result["date"].dtype)

(2721, 12)
['user_id', 'date', 'email', 'user_age_group', 'experiment_name', 'supplement_name', 'dosage_grams', 'is_placebo', 'average_heart_rate', 'average_glucose', 'sleep_hours', 'activity_level']
                                user_id       date             email  \
0  c6ae338a-9f95-481c-a88d-24a58bc8fc71 2018-01-31  hi_1@example.com   
1  c6ae338a-9f95-481c-a88d-24a58bc8fc71 2018-02-28  hi_1@example.com   
2  c6ae338a-9f95-481c-a88d-24a58bc8fc71 2018-03-31  hi_1@example.com   
3  c6ae338a-9f95-481c-a88d-24a58bc8fc71 2018-03-31  hi_1@example.com   
4  c6ae338a-9f95-481c-a88d-24a58bc8fc71 2018-04-30  hi_1@example.com   

  user_age_group experiment_name supplement_name  dosage_grams is_placebo  \
0        Over 65   Sleep Quality            Zinc      0.149315      False   
1        Over 65             NaN       No intake           NaN        NaN   
2        Over 65       Endurance       Magnesium      0.239949       True   
3        Over 65           Focus         Placebo      0.416